# Fusion

## 1. Imports et premières informations

In [89]:
import numpy as np
import pandas as pd

In [90]:
df_allocine = pd.read_csv("df_allocine.csv")
df_imdb = pd.read_csv("df_imdb.csv")

display(df_allocine.head())
print()
display(df_imdb.head())

,notesAllocine,avis,dates,review_id,series,allocine_id
0,4.5,Seule le format est décevant. Seulement 6 épis...,2026-02-08,282301,A Knight of the Seven Kingdoms,28230
1,5.0,Très bonne série. Des épisodes trop court tr...,2026-02-07,282302,A Knight of the Seven Kingdoms,28230
2,5.0,J’y ai trouvé du génie ! Pour ceux qui disent ...,2026-02-07,282303,A Knight of the Seven Kingdoms,28230
3,5.0,J'en suis au 4ème épisode et pour l'instant j'...,2026-02-06,282304,A Knight of the Seven Kingdoms,28230
4,5.0,"Encore un grand cru, un bijoux, du caviar, ou ...",2026-02-10,282305,A Knight of the Seven Kingdoms,28230


,notesImdb,avis,dates,series
0,8,Honest Opinion - Turns out Westeros is way mor...,2026-01-24,A Knight of the Seven Kingdoms
1,9,"So Far, So Good (Don't Screw It Up) - The firs...",2026-01-19,A Knight of the Seven Kingdoms
2,9,A proper return to Westeros - I stayed up for ...,2026-01-19,A Knight of the Seven Kingdoms
3,9,The only disappointment i had with ep 1 - Its ...,2026-01-19,A Knight of the Seven Kingdoms
4,9,Really freaking good with only one bad moment ...,2026-01-19,A Knight of the Seven Kingdoms


In [91]:
print(df_allocine.info())
print()
print(df_imdb.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 210 entries, 0 to 209
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   notesAllocine  210 non-null    float64
 1   avis           210 non-null    object 
 2   dates          210 non-null    object 
 3   review_id      210 non-null    int64  
 4   series         210 non-null    object 
 5   allocine_id    210 non-null    int64  
dtypes: float64(1), int64(2), object(3)
memory usage: 10.0+ KB
None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1412 entries, 0 to 1411
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   notesImdb  1412 non-null   int64 
 1   avis       1412 non-null   object
 2   dates      1412 non-null   object
 3   series     1412 non-null   object
dtypes: int64(1), object(3)
memory usage: 44.3+ KB
None


## 2. Concaténation et transformations

In [92]:
df_concat = pd.concat([df_allocine, df_imdb], axis=0)
df_concat

,notesAllocine,avis,dates,review_id,series,allocine_id,notesImdb
0,4.5,Seule le format est décevant. Seulement 6 épis...,2026-02-08,282301.0,A Knight of the Seven Kingdoms,28230.0,NaN
1,5.0,Très bonne série. Des épisodes trop court tr...,2026-02-07,282302.0,A Knight of the Seven Kingdoms,28230.0,NaN
2,5.0,J’y ai trouvé du génie ! Pour ceux qui disent ...,2026-02-07,282303.0,A Knight of the Seven Kingdoms,28230.0,NaN
3,5.0,J'en suis au 4ème épisode et pour l'instant j'...,2026-02-06,282304.0,A Knight of the Seven Kingdoms,28230.0,NaN
4,5.0,"Encore un grand cru, un bijoux, du caviar, ou ...",2026-02-10,282305.0,A Knight of the Seven Kingdoms,28230.0,NaN
...,...,...,...,...,...,...,...
1407,NaN,Mediocre filmmaking - Weak script in combinati...,2025-10-29,NaN,King & Conqueror,NaN,1.0
1408,NaN,drunk history - As many other reviewers have n...,2026-02-02,NaN,King & Conqueror,NaN,3.0
1409,NaN,I get it... -,2026-02-26,NaN,King & Conqueror,NaN,7.0
1410,NaN,Realism... - Earl's military crews count 5-8 s...,2026-03-02,NaN,King & Conqueror,NaN,3.0


In [93]:
# Création de la colonne note unique sur une même échelle
df_concat['notes'] = (df_concat['notesAllocine'].fillna(0) * 2).astype(int) \
    + df_concat['notesImdb'].fillna(0)

# Conservation du nom de la plateforme
df_concat['plateforme'] = np.where(df_concat["notesAllocine"].isna(), 'Imdb', 'Allocine')

# Dates en format 'date'
df_concat['dates'] = pd.to_datetime(df_concat['dates'])

# Supprimer les colonnes 'notesAllocine' et 'notesImdb' plus utiles
df_concat = df_concat.drop(['notesAllocine', 'notesImdb'], axis=1)

In [ ]:
# remplir les NaN de la colonne 'allocine_id'
df_concat['allocine_id'].groupby('series')['allocine_id'].ffill()

print(df_concat['allocine_id'].value_counts(), '\n')

# remplir les NaN de la colonne 'review_id'
for serie in df_concat['series'].unique():
    for i, review in enumerate(df_concat['review_id']):
        if pd.isna(review):
        # Retrouver la valeur de la ligne d'avant grâce à l'indice
            df_concat['review_id'].iloc[i] = df_concat['review_id'].iloc[i-1] + i


print(df_concat['review_id'].value_counts())
print(df_concat['review_id'])

allocine_id
28230.0    1404
35604.0     218
Name: count, dtype: int64 



C:\Users\CLFO0303\AppData\Local\Temp\ipykernel_23292\3859837935.py:15: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_concat['review_id'].iloc[i] = df_concat['review_id'].iloc[i-1] + i
C:\Users\CLFO0303\AppData\Local\Temp\ipykernel_23292\3

review_id
282301.0     1
282302.0     1
282303.0     1
282304.0     1
282305.0     1
            ..
4846653.0    1
4848271.0    1
4849890.0    1
4851510.0    1
4853131.0    1
Name: count, Length: 1622, dtype: int64


C:\Users\CLFO0303\AppData\Local\Temp\ipykernel_23292\3859837935.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_concat['review_id'].iloc[i] = df_concat['review_id'].iloc[i-1] + i
C:\Users\CLFO0303\AppData\Local\Temp\ipykernel_23292\3859837935.py:15: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = val

In [96]:
# Export en csv
df_concat.to_csv('df_clean.csv', index=False)